# Proximal Policy Optimization

$\theta$: parameters of the policy network; and the policy can be defined as: $\pi_{\theta}(a|s)$, which equals the probability of taking action $a$ in state $s$.
1. For discrete actions: the network gives logits $\longrightarrow$ softmax $\longrightarrow$ probabilities
2. For continous actions: network gives: mean and log($\sigma$) of the Gaussian distrbution of (action probabilities)

Let's have the cost function as the expected return under policy $\pi(\theta)$:

$$ J(\theta) = \mathbb{E}_{\pi_{\theta}}(\mathcal{G}_t) = \mathbb{E}_{\pi_{\theta}}(R_t + \gamma \mathcal{G}_{t+1}) $$

Now, we know return $\mathcal{G}(t)$ depends on which states the MDP visits, which is directly dictated by the underlying system dynamics as well as the policy $\pi$. Let $\tau$ be such a trajectory, hence we can write the following:

$$ J(\theta) = \sum_{\forall \tau} p(\tau | \theta) \mathcal{G}(\tau) $$

$\tau$ is the set of trajectories and any one trajectory can be defined as: $(s_0, a_0, r_0, s_1, a_1, r_1, ..., s_n, a_n, r_n)$ followed when using policy $\pi_{\theta}$. And $p(\tau | \theta)$ is the probability to follow trajectory $\tau$ under policy $\pi_{\theta}$.

# Probability of Following a trajectory $\tau$

$$ P(\tau | \theta) = p(s_0) \prod^{T-1}_{t=0} \pi_{\theta} (a_t | s_t) \times p(s_{t+1} | s_t, a_t) $$

Since $J(\theta)$ is the expected value of return, we want to maximise $J$. We use gradient descent with respect to $\theta$ because we need to find:

$$ \theta': \theta' = \text{argmax}_{\theta}J(\theta) $$

Taking gradient of $J(\theta)$ we get, 

$$ \nabla_{\theta} J(\theta) = \nabla_{\theta} \mathbb{E}_{\pi_{\theta}} \mathcal{G}(\tau) $$

$$ \nabla_{\theta} J(\theta) = \nabla_{\theta} \sum_{\forall \tau} p(\tau | \theta) \mathcal{G}(\tau) $$

Converting summation to integral by considering infinitisimal variations among $\tau$:

$$ \nabla_{\theta} J(\theta) = \nabla_{\theta} \int_{\tau} p(\tau | \theta) \mathcal{G}(\tau) d\tau$$

Using the $\nabla_{\theta} p = p \nabla_{\theta} log(p)$ trick, and taking the $\nabla$ inside the integral we get,

$$ \nabla_{\theta} J(\theta) = \int_{\tau} \nabla_{\theta} p(\tau | \theta) \mathcal{G}(\tau) d\tau$$

$$ \nabla_{\theta} J(\theta) = \int_{\tau} p(\tau | \theta) \nabla_{\theta} log (p(\tau | \theta)) \mathcal{G}(\tau) d\tau $$

Now, let's solve for $\nabla_{\theta} log (p(\tau | \theta))$:

$$ \nabla_{\theta} log (p(\tau | \theta)) = \nabla_{\theta} log \left\{ p(s_0) \prod^{T-1}_{t=0} \pi_{\theta} (a_t | s_t) \times p(s_{t+1} | s_t, a_t)) \right\} $$

$$ \nabla_{\theta} log (p(\tau | \theta)) = \nabla_{\theta} \left\{ log p(s_0) + \sum_{t=0}^{T-1} \left\{ log(\pi_{\theta} (a_t | s_t)) + log(p(s_{t+1} | s_t, a_t)) \right\} \right \}$$

The terms which don't depend on $\theta$ their derivative is going to be zero, hence,

$$ \nabla_{\theta} log (p(\tau | \theta)) = \nabla_{\theta} \sum_{t=0}^{T-1} log(\pi_{\theta} (a_t | s_t)) $$

So, what does this say? Take the gradient of the sum of log of all action probabilities at all times for that particular trajectory. Now, substitute this back in the integral we found earlier: 

$$ \nabla_{\theta} J(\theta) = \int_{\tau} p(\tau | \theta) \nabla_{\theta} log (p(\tau | \theta)) \mathcal{G}(\tau) d\tau $$

This structure when morphed back to the Expected value form looks like:

$$ \nabla_{\theta} J(\theta) = \mathbb{E}_{\pi_{\theta}} \left \{ \nabla_{\theta} log (p(\tau | \theta)) \mathcal{G}(\tau) \right \} $$

$$ \nabla_{\theta} J(\theta) = \mathbb{E}_{\tau \sim \pi_{\theta}} \left \{ \left\{ \sum_{t=0}^{T-1} \nabla_{\theta} log(\pi_{\theta} (a_t | s_t)) \right \} \mathcal{G}(\tau) \right \} $$

## Replacing $G(\tau)$ with $Q^{\pi}(s, a)$

Look at the expression above, it says, take the expected value over all the set of trajectories, which are reachable through when following a policy $\pi_{\theta}$. Let's consider the trajectories to be $\tau:= \tau_1, \tau_2, \tau_3, \dots, \tau_n $ and so on.

$$ \tau_1 := \{s_{(0, 1)}, a_{(0, 1)}, r_{(0, 1)}, s_{(1, 1)}, a_{(1, 1)}, r_{(1, 1)}, s_{(2, 1)}, a_{(2, 1)}, r_{(2, 1)}, \dots, s_{(n, 1)}, a_{(n, 1)}, r_{(n, 1)} \} $$

Now, what is the probability of encountering this particular trajectory, when following the policy $\pi_{\theta}$:

 $$ p(\tau = \tau_1 | \theta) = p(s_0) \prod^{T-1}_{t=0} \pi_{\theta}(a_t | s_t) p(s_{t+1} | s_t, a_t) $$

Coming back to, $\nabla_{\theta} J(\theta)$,

$$ \nabla_{\theta} J(\theta) = 
\sum^{T}_{t=0} \sum_{\forall \tau} p(\tau | \theta) \nabla log \pi_{\theta} (a_t | s_t) G(t) \times \mathbb{1}_{t \leq T} $$ 

The end term $\mathbb{1}_{t \leq T}$ is added to make sure the sum is finite, because in usual cases: $T \longrightarrow \infty $

